# 🧰 10.20 Built-in Functions vs UDFs

Welcome to **SECTION G: CUSTOM LOGIC**!

In the previous sections, we learned how to use PySpark's massive library of **Built-in Functions** (like `F.sum()`, `F.split()`, and `F.to_date()`). These built-in functions are incredibly fast because they are highly optimized by Spark's internal engines.

But what happens when you need to apply a highly specific, proprietary business rule that Spark doesn't have a built-in function for? What if you want to run a custom Python algorithm on every single row of your dataset?

To do this, we use **User-Defined Functions (UDFs)**. However, UDFs come with a massive, hidden performance penalty. In this lesson, we will learn how to create them, why they are dangerous, and how to avoid them whenever possible.

### 🎯 Learning Objectives
* Understand **Why** and **When** to use User-Defined Functions (UDFs).
* Learn how to create and register a Python UDF in PySpark.
* Understand the massive **Performance Impact** of Python Serialization.
* Discover **Alternatives to UDFs** using PySpark's native functions.
* See why minimizing UDFs is a key differentiator between Junior and Senior Data Engineers.

In [ ]:
# 0. Setup: Let's start our SparkSession and create some dummy data.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("UDF_Lesson").master("local[*]").getOrCreate()

# Imagine we have a table of users and their credit scores.
data = [
    (1, "Alice", 780),
    (2, "Bob", 590),
    (3, "Charlie", 650),
    (4, "Diana", 810)
]
columns = ["user_id", "name", "credit_score"]

df = spark.createDataFrame(data, columns)
print("Raw Customer Data:")
df.show()

## 1. Why UDFs?

A **User-Defined Function (UDF)** allows you to write standard, everyday Python code and apply it to a distributed Spark DataFrame.

**When do we use them?**
You use UDFs when your logic is too complex for standard SQL or PySpark built-in functions. 
For example:
* You have a massive, proprietary Python library that calculates a custom "Risk Score" using complex math.
* You need to make a network request to an external API for every row.
* You have complex string parsing logic that requires advanced Python libraries (like `nltk` for natural language processing).

## 2. Creating UDFs

Creating a UDF requires three simple steps:
1. **Write a standard Python function:** Write normal Python code that takes a single input and returns a single output.
2. **Register it as a UDF:** Wrap your Python function using `F.udf()`, and tell Spark exactly what data type it will return (e.g., String, Integer).
3. **Apply it:** Use it inside a `.withColumn()` or `.select()` statement just like a normal Spark function.

Let's write a UDF to categorize credit scores into Risk Tiers.

In [ ]:
from pyspark.sql.types import StringType

print("--- Creating and Applying a UDF ---")

# Step 1: Write standard Python logic
def calculate_risk_tier(score):
    if score >= 750:
        return "Low Risk"
    elif score >= 600:
        return "Medium Risk"
    else:
        return "High Risk"

# Step 2: Register it as a Spark UDF. 
# We MUST tell Spark what type of data this function returns (StringType)!
risk_tier_udf = F.udf(calculate_risk_tier, StringType())

# Step 3: Apply the UDF to our DataFrame
df_with_risk = df.withColumn("risk_tier", risk_tier_udf(F.col("credit_score")))
df_with_risk.show()

## 3. The Performance Impact (The "Python Penalty")

The code above worked perfectly! It looks clean and easy to read. So why do Data Engineers hate UDFs?

Remember **Lesson 10.9 (Why DataFrames Replaced RDDs)**? We learned about the **Tungsten Engine**. Spark runs in the Java Virtual Machine (JVM), and Tungsten processes data natively in binary memory. This is lightning fast.

When you use a Python UDF, Spark cannot execute it in the JVM. It is forced to do the following for **every single row**:
1. Pause the fast JVM execution.
2. Serialize (translate) the data from the JVM into Python.
3. Send the data to a Python process to run your custom function.
4. Serialize the result back from Python into the JVM.

This translation process is called the **Serialization Penalty**. Furthermore, the **Catalyst Optimizer** is completely blind to UDFs. It treats your Python code like a black box and cannot optimize it at all.

<img src="../assets/Module_10/10_20_01.png" alt="A visual comparing a fast train (Built-in Spark Functions) running smoothly on continuous tracks, versus a car constantly stopping at toll booths (Python UDFs) representing the serialization penalty between the Java Virtual Machine and the Python interpreter." width="75%">

## 4. Alternatives to UDFs

Because of the extreme performance penalty, the Golden Rule of PySpark is:
> 🌟 **Always use Built-in Spark Functions instead of Python UDFs whenever physically possible.** 🌟

Spark's built-in `pyspark.sql.functions` library is massive. 99% of the time, you can replicate your Python logic using native Spark functions.

Let's rewrite our `calculate_risk_tier` logic using Spark's native `when().otherwise()` functions. This executes entirely inside the JVM and runs 10x to 100x faster than our UDF!

In [ ]:
print("--- The Native, Optimized Way (No UDFs) ---")

# We use F.when() to act like an 'if' statement, and .otherwise() to act like 'else'
df_native_optimized = df.withColumn(
    "risk_tier",
    F.when(F.col("credit_score") >= 750, "Low Risk")
     .when(F.col("credit_score") >= 600, "Medium Risk")
     .otherwise("High Risk")
)

df_native_optimized.show()
print("Result is identical, but this runs at lightning-fast Java/Tungsten speeds!")

### What if I *MUST* use Python?
Sometimes, you simply cannot use built-in functions. If your company relies on a massive Machine Learning model written in Python, you have to use Python.

In these cases, modern Spark offers **Pandas UDFs (Vectorized UDFs)**. 
Instead of sending data from the JVM to Python *row-by-row*, Pandas UDFs use Apache Arrow to send data in massive *batches*. This significantly reduces the serialization penalty, making them much faster than traditional row-by-row Python UDFs.

## 5. Why UDFs Should Be Avoided When Possible

Let's summarize why native functions always win:
1. **Execution Speed:** Native functions run in binary/JVM (Tungsten Engine); UDFs suffer the Python translation penalty.
2. **Optimization:** Catalyst Optimizer can rewrite and speed up native functions. UDFs are "black boxes" that Catalyst cannot see inside.
3. **Memory:** Serialization creates copies of your data in memory. Heavy UDF usage is a leading cause of Out-Of-Memory (OOM) crashes in Spark.

## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

How do different roles approach custom logic?

| Feature | 🏛️ Traditional DBA | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Custom Logic** | Writes SQL User-Defined Functions (PL/pgSQL). | **Chains dozens of native PySpark functions together to avoid Python UDFs.** | Loves Python UDFs because they can use their favorite libraries (numpy/scipy) directly on Spark rows. |
| **Performance View** | Depends on database indexes. | **Obsesses over JVM vs Python boundaries. Rewrites Data Scientist UDFs into native PySpark functions.** | Doesn't mind if a job takes 2 hours as long as the model logic works perfectly. |
| **Modern Tools** | Stored Procedures | **Native PySpark SQL Functions & Pandas UDFs (Apache Arrow).** | Pandas `apply()` function. |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Falls back to Python UDFs immediately because writing standard Python `if/else` statements feels easier than learning PySpark's specific syntax. Wonders why their pipelines take 5 hours to run.
2. **Mid-Level DE (Your Current Phase!):** Actively avoids UDFs. Becomes an expert in the `pyspark.sql.functions` library. Uses `F.when()`, `F.split()`, and `F.regexp_replace()` to keep all execution inside the fast Tungsten Engine.
3. **Senior DE:** Masters **Pandas Vectorized UDFs**. When forced to integrate a Data Scientist's complex Python Machine Learning model into a massive Spark pipeline, the Senior DE wraps it in a Pandas UDF to ensure it processes data in large batches (via Apache Arrow) rather than row-by-row.

### 🛠️ Where Data Engineering Fits in Modern Organizations
Performance tuning is the defining trait of Data Engineering. Anyone can write code that works on 1 Megabyte of data. The Data Engineer is hired to write code that works on 10 Petabytes of data without crashing the company's servers or running up a $100,000 AWS bill.

## 🔑 Key Takeaways

- **UDFs (User-Defined Functions):** Allow you to write custom Python code and apply it to a distributed Spark DataFrame.
- **The Python Penalty:** Traditional Python UDFs are extremely slow because Spark must translate data row-by-row between the Java Virtual Machine (JVM) and Python.
- **The Black Box:** The Catalyst Optimizer cannot understand Python code, meaning it cannot optimize your UDFs.
- **The Golden Rule:** Always use Spark's built-in functions (like `F.when()`) instead of UDFs whenever possible to keep execution native and lightning fast.
- **Pandas UDFs:** If you absolutely *must* use Python, use Pandas UDFs (Vectorized) to process data in batches rather than row-by-row.

## 📝 Knowledge Check Quiz

**Question 1:** Why is a native PySpark function like `F.when()` significantly faster than writing a custom Python `if/else` statement inside a UDF?
A) Because Python is a compiled language.
B) Because native PySpark functions run directly inside the highly optimized Java Virtual Machine (Tungsten Engine), avoiding the massive overhead of serializing data back and forth to a Python interpreter.
C) Because UDFs can only process one partition at a time.
D) Because native functions automatically drop missing data.

**Question 2:** How does the Catalyst Optimizer treat Python UDFs?
A) It translates them perfectly into Java.
B) It automatically converts them to SQL.
C) It treats them as a "black box" and cannot optimize the logic inside them at all.
D) It deletes them if they are too slow.

**Question 3:** A Data Scientist hands you a complex Python Machine Learning algorithm that uses custom Python libraries. It is impossible to rewrite this using built-in Spark functions. What is the most optimized way to run this Python code on Spark?
A) Run it on a single machine using standard Python.
B) Wrap the logic in a traditional Row-by-Row Python UDF.
C) Wrap the logic in a **Pandas UDF (Vectorized UDF)** to process the data in massive batches using Apache Arrow, minimizing serialization overhead.
D) Convert the PySpark DataFrame back into an RDD.

---

*Answers: 1: B, 2: C, 3: C*

In [ ]:
# Clean up our session
spark.stop()
print("Spark session closed.")

### 🚀 What's Next?
Congratulations! You have officially conquered **Section G** and mastered the logic and APIs of PySpark.

But there is one final, crucial piece to the puzzle. Our pipelines are running perfectly in memory, but eventually, we have to save our results back to the hard drive. 

In **SECTION H: STORAGE & I/O**, we will learn how to read and write data to the disk. More importantly, we will dive into the most famous file format in Big Data: **Apache Parquet**. See you in **10.21 Reading Data Sources**!